# 03: Carry (Twist-Only): Walk while Holding a Box, driven by twist like walk

| | |
|---|---|
| **Project** | Humanoid Loco-Manipulation with the Unitree G1 |
| **Developer** | Narayane Ribeiro Medeiros |
| **Repository** | [https://github.com/narayanerm/Humanoid-Loco-Manipulation](https://github.com/narayanerm/Humanoid-Loco-Manipulation) |

Trains a **G1 humanoid** to walk to a randomised goal position while holding a box between its hands.

### Run-Mode Flags

Boolean switches: reuse a carry checkpoint, resume, warm-start from the walk
policy, freeze the transferred trunk for a diagnostic run, run a smoke test,
or open the viewer.

In [ ]:
REUSE_TRAINING       = False    # skip training if PPO checkpoint exists
RESUME               = False    # continue from the most recent PPO checkpoint
WARM_START_FROM_WALK = True     # load walk checkpoint as initialization (hierarchical-style HRL)
DIAGNOSTIC_TRUNK_FROZEN = False # if True, freeze transferred trunk (mlp.2, mlp.4) and  replace head (mlp.6)
WARM_START_FROM_PICK = True     # True: (re)use outputs/pick_terminal_states.npz, generating from a pick ckpt if missing. False: skip generation; assumes the buffer is already in place.
DEBUG_LIGHT          = True     # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER           = True     # open viser viewer at localhost:8080

# Multi-seed aggregation. When True, the training cell loops N_SEEDS runs
# (sequentially, same GPU) and the plot cell renders mean +/- std curves.
# When False, the notebook runs a single training and plots the raw curves.
MULTI_SEED           = False
N_SEEDS              = 10
SEED_LIST            = None     # None -> draw N_SEEDS random ints (printed at run time); else explicit list


### Environment Setup

Installs pinned dependencies from `config/requirements.txt` via **uv** (5-10x
faster than pip, idempotent), configures MuJoCo in headless mode (EGL), and
detects the training GPU.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass


### Robot Initialization

Loads `TASK_ID`, G1 joints/home pose, and carry-specific constants (box
geometry, chest height, goal range) from `config/policy_carry.py`. Same
pattern as walk/pick.

In [ ]:
import sys, os
from pathlib import Path
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))

from src.policy_carry import TASK_ID
from src.policy_carry import (
    G1_JOINTS, G1_HOME_ROOT_POS, G1_HOME_ROOT_ROT, G1_HOME_JOINT_POS,
    BOX_HALF, BOX_HOLD_X_OFFSET, BOX_HOLD_Z, BOX_DROP_Z,
    BOX_MASS_MIN, BOX_MASS_MAX,
    CHEST_Z,
)

print(f'Task ID           : {TASK_ID}')
print(f'Joints            : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home root z       : {G1_HOME_ROOT_POS[2]:.3f} m')
print()
print(f'Box start (front) : {BOX_HOLD_X_OFFSET:.2f} m in front of trunk, z={BOX_HOLD_Z:.2f} m')
print(f'Box drop limit    : z < {BOX_DROP_Z:.2f} m to termination (dropped)')
print(f'Box mass DR range : [{BOX_MASS_MIN:.2f}, {BOX_MASS_MAX:.2f}] kg')
print()


### Task Config & Registration

Full MDP (observations, actions, rewards, events, terminations, sensors) and
the PPO runner cfg live in **`config/policy_carry.py`**. Edit that file to
change rewards, box/goal geometry, DR events, or PPO hyperparameters, then
re-run this cell.

In [ ]:
from src import policy_carry

policy_carry.register()

env_cfg_dbg = policy_carry.g1_carry_env_cfg(play=False)
print(f'Task registered   : {policy_carry.TASK_ID}')
print(f'Entities          : {list(env_cfg_dbg.scene.entities.keys())}')
print(f"Sensors           : {len(env_cfg_dbg.scene.sensors)}  ({', '.join(s.name for s in env_cfg_dbg.scene.sensors)})")
print(f'Events            : {list(env_cfg_dbg.events.keys())}')
print(f'Rewards           : {len(env_cfg_dbg.rewards)} terms   to {list(env_cfg_dbg.rewards.keys())}')
print(f'Terminations      : {list(env_cfg_dbg.terminations.keys())}')
print(f'Curriculum        : {list(env_cfg_dbg.curriculum.keys())}')
print(f'Sim iter/ls       : {env_cfg_dbg.sim.mujoco.iterations}/{env_cfg_dbg.sim.mujoco.ls_iterations}')
print(f'Episode length    : {env_cfg_dbg.episode_length_s:.2f} s   ({int(env_cfg_dbg.episode_length_s / 0.025)} steps @ 40 Hz)')


### Simulation Parameters

Global run settings shared by training and play: task, parallel envs, PPO
hyperparameters. `DEBUG_LIGHT` shrinks to 64 envs / 3 iters for a smoke test.

In [ ]:
from pathlib import Path
import sys; root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as os_mod; os_mod.chdir(root); sys.path.insert(0, str(root))
from src.utils import find_latest_checkpoint

OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media'; MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT      = 'g1_carry'
WALK_EXPERIMENT_HINT = 'g1_walk'  # source of the walk checkpoint used for warm-start
LOG_ROOT             = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 1000

SIM_TIMESTEP     = 0.005
DECIMATION       = 5
EPISODE_LENGTH_S = 8.0

LEARNING_RATE    = 3e-4
CLIP_PARAM       = 0.2
GAMMA            = 0.99
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Episode length  : {EPISODE_LENGTH_S:.2f} s')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')


### Walk-Terminal Buffer

The carry reset (inherited from pick) samples each episode from a buffer of
walk-terminal states so the robot spawns in a walking-realistic pose and
velocity: critical for warm-start, because the transferred actor was trained
on that state distribution. Reuses `outputs/walk_terminal_states.npz` if
present, otherwise rolls the trained walk policy across parallel envs and
snapshots. Requires a walk checkpoint in `outputs/logs/g1_velocity/` (run
`01_walk.ipynb` first).

In [ ]:
import subprocess, sys
from pathlib import Path
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as os_mod; os_mod.chdir(root); sys.path.insert(0, str(root))
from src.utils import find_latest_checkpoint
from src.policy_carry import PICK_BUFFER_PATH

BUFFER_PATH   = Path(PICK_BUFFER_PATH).resolve()
BUFFER_N      = 20_000
BUFFER_ENVS   = 2048 if not DEBUG_LIGHT else 128

if not WARM_START_FROM_PICK:
    if not BUFFER_PATH.exists():
        print(f'[warn] WARM_START_FROM_PICK=False and no buffer at {BUFFER_PATH}. '
              f'Training will crash on the first reset. Copy a pick_terminal_states.npz here or set WARM_START_FROM_PICK=True.')
    else:
        print(f'[skip] WARM_START_FROM_PICK=False; using existing buffer at {BUFFER_PATH}')
elif BUFFER_PATH.exists():
    print(f'[reuse] pick buffer already at {BUFFER_PATH}')
else:
    pick_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint='g1_pick')
    if pick_ckpt is None:
        raise RuntimeError(
            'No pick checkpoint found in outputs/logs/g1_pick and no buffer to reuse. '
            'Options: train pick first (01a_pick.ipynb), copy an existing pick_terminal_states.npz into outputs/, '
            'or set WARM_START_FROM_PICK=False if you have arranged the buffer another way.'
        )
    print(f'Pick ckpt        : {pick_ckpt}')
    print(f'Collecting {BUFFER_N} pick-terminal states across {BUFFER_ENVS} envs...')
    subprocess.check_call([
        sys.executable, '-m', 'src.collect_buffer',
        '--policy-module', 'src.policy_pick',
        '--checkpoint', str(pick_ckpt),
        '--out',        str(BUFFER_PATH),
        '--num-envs',   str(BUFFER_ENVS),
        '--n-snapshots', str(BUFFER_N),
    ])
    print(f'Buffer written   : {BUFFER_PATH}')


### Training Setup

Builds `TrainConfig` from the registered task and applies notebook overrides
(envs, iterations, logger). If `WARM_START_FROM_WALK=True`, the setup points
at the latest `g1_walk` checkpoint and uses the shape-lenient `load`
(`strict=False`, actor+critic only) so walk weights transfer to the first
carry-actor layers even when obs dimensionality changes (new dims start silent).

In [ ]:
import datetime, os, secrets
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False

run_stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
log_dir_root = Path(LOG_ROOT) / EXPERIMENT_HINT / run_stamp
log_dir = log_dir_root

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('MUJOCO_GL', 'egl')

if MULTI_SEED:
    if SEED_LIST is not None:
        _seeds = list(SEED_LIST)
        _seed_source = 'SEED_LIST (user-provided)'
    else:
        _rng = secrets.SystemRandom()
        _seeds = _rng.sample(range(1, 2**31 - 1), N_SEEDS)
        _seed_source = 'random (secrets.SystemRandom)'
    _mode = f'MULTI-SEED ({len(_seeds)} runs)'
else:
    _seeds = [getattr(cfg.agent, 'seed', 42)]
    _seed_source = 'cfg.agent.seed'
    _mode = ('RESUME' if RESUME
             else 'WARM_START_FROM_WALK' if WARM_START_FROM_WALK
             else 'FROM SCRATCH')

print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'Mode            : {_mode}')
print(f'Seed source     : {_seed_source}')
print(f'Seeds queued    : {_seeds}')
print(f'Log dir root    : {log_dir_root.resolve()}')


### Run Training

Blocks the kernel until PPO training finishes. Use "Interrupt Kernel" to stop.

In [ ]:
import gc
import sys, os
from pathlib import Path

import torch

for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from src.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path


def _vram_snapshot(tag):
    """Print live + cached VRAM in GB so leaks across seeds are visible."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f'  [VRAM {tag:12s}] live={alloc:.2f} GB  cached={reserv:.2f} GB  peak={peak:.2f} GB')


def _hard_cuda_cleanup():
    """Flush orphaned CUDA state between training runs."""
    for _ in range(2):
        gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


# Warm-start walk checkpoint shared across all seeds.
_warm_ckpt_shared = None
if WARM_START_FROM_WALK:
    _warm_ckpt_shared = find_latest_checkpoint(LOG_ROOT, experiment_hint=WALK_EXPERIMENT_HINT)
    if _warm_ckpt_shared is None:
        raise RuntimeError(
            f'WARM_START_FROM_WALK=True but no walk checkpoint found at '
            f'{LOG_ROOT}/{WALK_EXPERIMENT_HINT}. Run 01_walk.ipynb first.'
        )

if MULTI_SEED:
    _vram_snapshot('baseline')

    for _i, _seed in enumerate(_seeds):
        _log_dir = log_dir_root / f'seed{_seed}'
        cfg.agent.seed = _seed
        cfg.agent.resume = False

        print(f'\n=== seed {_seed} ({_i + 1}/{len(_seeds)}) -> {_log_dir} ===')
        _vram_snapshot('pre-train')

        run_train_loop(
            task_id=TASK_ID,
            cfg=cfg,
            log_dir=_log_dir,
            warm_start_ckpt=_warm_ckpt_shared,
            resume_ckpt=None,
            diagnostic_trunk_frozen=DIAGNOSTIC_TRUNK_FROZEN,
        )

        _vram_snapshot('post-train')
        _hard_cuda_cleanup()
        _vram_snapshot('post-clean')

    print(f'\n[MULTI-SEED] finished {len(_seeds)} runs under {log_dir_root}')
    print('If `live` or `cached` VRAM climbed monotonically across iterations,')
    print('the loop is leaking and you should run one seed per subprocess instead.')
else:
    existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
    if REUSE_TRAINING and existing_ckpt is not None:
        print(f'[REUSE] Checkpoint found: skipping training.\n        {existing_ckpt}')
    else:
        resume_ckpt = None
        if RESUME:
            resume_ckpt = get_checkpoint_path(
                log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
            )

        run_train_loop(
            task_id=TASK_ID,
            cfg=cfg,
            log_dir=log_dir,
            warm_start_ckpt=_warm_ckpt_shared,
            resume_ckpt=resume_ckpt,
            diagnostic_trunk_frozen=DIAGNOSTIC_TRUNK_FROZEN,
        )


### Training Curves & Robustness

Post-training analysis in two blocks: (1) **training curves** parsed from the TensorBoard event file: reward, PPO diagnostics, per-term reward decomposition, terminations and command tracking; (2) **robustness**: headless rollouts of the trained policy under lateral pushes and a commanded-velocity sweep, following the RSL "Learning to Walk in Minutes" evaluation protocol (Rudin et al., CoRL 2022).

Set `RUN_ROBUSTNESS = False` in the setup cell below to skip figures 5-6 (the rollout-based ones).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as os_mod; os_mod.chdir(root); sys.path.insert(0, str(root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

RUN_ROBUSTNESS = True  # False skips Figs 5-6 (rollout-based)

COLORS = {
    'primary':   '#0072B2',
    'accent':    '#D55E00',
    'ok':        '#009E73',
    'warn':      '#E69F00',
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a scalar time-series from a TensorBoard event file."""
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List scalar tags filtered by prefix."""
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Centered moving-average filter."""
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Enable major/minor grid with auto minor locators."""
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)


def resolve_tb_dirs(log_root, experiment_hint):
    """Return the TB directories to plot (single-seed: [latest]; multi-seed: list of seed*/ subdirs)."""
    latest = find_latest_tb_log(log_root, experiment_hint=experiment_hint)
    if latest is None:
        return []
    latest = Path(latest)
    direct_events = any(p.name.startswith('events.out') for p in latest.iterdir() if p.is_file())
    if direct_events:
        return [latest]
    seed_dirs = sorted(
        [p for p in latest.iterdir() if p.is_dir() and p.name.startswith('seed')]
    )
    return seed_dirs if seed_dirs else [latest]


def agg_scalar(tb_dirs, tag):
    """Aggregate a scalar tag across TB directories.
    Returns (steps, mean, std, n) where mean/std are over seeds at each step.
    """
    series = []
    for d in tb_dirs:
        xs, ys = load_scalar(d, tag)
        if len(ys):
            series.append((xs, ys))
    if not series:
        return np.array([]), np.array([]), np.array([]), 0
    min_len = min(len(ys) for _, ys in series)
    xs_ref = series[0][0][:min_len]
    mat = np.stack([ys[:min_len] for _, ys in series], axis=0)
    mean = mat.mean(axis=0)
    std = mat.std(axis=0, ddof=1) if mat.shape[0] > 1 else np.zeros_like(mean)
    return xs_ref, mean, std, mat.shape[0]


TB_DIRS = resolve_tb_dirs(LOG_ROOT, EXPERIMENT_HINT)
assert TB_DIRS, 'No TensorBoard log found: train first.'
TB_DIR = TB_DIRS[0]
if len(TB_DIRS) == 1:
    print(f'TB dir   : {TB_DIR}')
else:
    print(f'TB group : {TB_DIRS[0].parent}   ({len(TB_DIRS)} seeds)')
    for d in TB_DIRS:
        print(f'  - {d.name}')


### Return & PPO Diagnostics

Plots training curves from the TensorBoard event file: mean reward, episode
length, reward gain dR/di, entropy, value loss, and policy std. Depends on
helpers defined above.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))


def plot_agg(ax, tag, color, log=False, label=None):
    """Plot across-seed mean of `tag` with a shaded +/- std band."""
    xs, mu, sd, n = agg_scalar(TB_DIRS, tag)
    if not len(xs):
        return xs, mu, sd, n
    mu_s = smooth(mu)
    if n > 1:
        sd_s = smooth(sd)
        ax.fill_between(xs, mu_s - sd_s, mu_s + sd_s,
                        color=color, alpha=0.20, linewidth=0)
    legend_label = label if label is not None else (f'mean (n={n})' if n > 1 else None)
    ax.plot(xs, mu_s, color=color, lw=1.2, label=legend_label)
    if log:
        ax.set_yscale('log')
    return xs, mu, sd, n


x_r, mu_r, sd_r, n_seeds = plot_agg(axes[0, 0], 'Train/mean_reward', COLORS['primary'])
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
if n_seeds > 1:
    axes[0, 0].legend(loc='lower right', frameon=False)
apply_grid(axes[0, 0])

_, mu_l, _, _ = plot_agg(axes[0, 1], 'Train/mean_episode_length', COLORS['ok'])
axes[0, 1].axhline(1000, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
if len(x_r):
    axes[0, 1].text(x_r[-1], 1010, 'max horizon', ha='right', va='bottom',
                    fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

if len(mu_r) > 1:
    r_smooth = smooth(mu_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

plot_agg(axes[1, 0], 'Loss/entropy',    COLORS['primary'])
axes[1, 0].set_xlabel('PPO iteration')
axes[1, 0].set_ylabel(r'Entropy $H(\pi)$')
apply_grid(axes[1, 0])

plot_agg(axes[1, 1], 'Loss/value',      COLORS['accent'], log=True)
axes[1, 1].set_xlabel('PPO iteration')
axes[1, 1].set_ylabel(r'Value loss $L_V$')
apply_grid(axes[1, 1])

plot_agg(axes[1, 2], 'Policy/mean_std', COLORS['ok'])
axes[1, 2].set_xlabel('PPO iteration')
axes[1, 2].set_ylabel(r'Policy std $\sigma_\pi$')
apply_grid(axes[1, 2])

if n_seeds > 1:
    fig.suptitle(
        f'Carry training curves  (n = {n_seeds} seeds, '
        f'shaded band = $\\pm$1 std across seeds)',
        fontsize=10,
    )
fig.tight_layout()
plt.savefig(MEDIA / 'carry_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

if len(mu_r):
    final_mean = float(mu_r[-100:].mean())
    if n_seeds > 1:
        final_std = float(sd_r[-100:].mean())
        print(f'Final 100 iters mean reward (across {n_seeds} seeds): '
              f'{final_mean:.2f} +/- {final_std:.2f}')
    else:
        print(f'Final 100 iters mean reward (single run): {final_mean:.2f}')


### Run Viewer

Blocks the kernel while the viewer is open. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'   # @param ["trained","random","zero"]
PLAY_NUM_ENVS       = 1           # @param {type:"integer"}
PLAY_VIEWER         = 'viser'     # @param ["viser","native"]
CHECKPOINT_OVERRIDE = ''          # @param {type:"string"}  leave blank for latest

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')


if RUN_VIEWER:
    print('Viewer starting: open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')